# 05 - What the model is really learning

## Why this notebook exists

Notebook 04 produced a model with AUC 0.825 and then a puzzle: adding 141
biologically-motivated features changed nothing. That is strange. Either the
features are useless, or **the model was never using weather at all** and there was
nothing for them to improve.

This notebook finds out which, using three tools:

1. **Negative controls** - break the thing the model is supposed to use, and see
   whether the score drops. If it does not drop, the model was not using it.
2. **Decomposition** - separate "which region is this?" from "which week is this?"
3. **Within-region evaluation** - score each region on its own, which is the only
   question a real surveillance system has to answer.

This is the most important notebook in the project. Everything else follows from it.

In [1]:
import sys, csv, math, warnings
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, average_precision_score
warnings.filterwarnings("ignore")
rng = np.random.default_rng(7)

ROOT = Path.cwd()
while not (ROOT / "data").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
TABLE = ROOT / "data" / "processed" / "taiwan_model_table_v2.csv"
META = {"region", "year", "week", "yearweek", "cases"}

rows = list(csv.DictReader(open(TABLE)))
feats = [c for c in rows[0] if c not in META]
X = np.array([[float(r[c]) for c in feats] for r in rows])
y = np.array([1 if int(r["cases"]) > 0 else 0 for r in rows])
yr = np.array([int(r["year"]) for r in rows])
wk = np.array([int(r["week"]) for r in rows])
reg = np.array([r["region"] for r in rows])
REGIONS = sorted(set(reg.tolist()))
FOLDS = list(range(2012, 2026))

def ET(sd=0):
    return ExtraTreesClassifier(n_estimators=400, min_samples_leaf=2,
        class_weight="balanced_subsample", n_jobs=-1, random_state=sd)

print(f"{len(rows)} region-weeks, {len(feats)} features, {int(y.sum())} positive weeks")
for r in REGIONS:
    m = reg == r
    print(f"  {r:10s} base rate {y[m].mean()*100:5.1f}%   positive weeks {int(y[m].sum()):4d}")

7362 region-weeks, 185 features, 614 positive weeks
  Central    base rate   6.8%   positive weeks   84
  Eastern    base rate   0.4%   positive weeks    5
  Kaoping    base rate  31.4%   positive weeks  385
  Northern   base rate   1.4%   positive weeks   17
  Southern   base rate   8.0%   positive weeks   98
  Taipei     base rate   2.0%   positive weeks   25


## Part 1 - Negative controls

A negative control is a deliberately sabotaged version of the experiment. If the
model still performs after you destroy its information source, that source was not
the information it was using.

Five sabotages:

| control | what it destroys | what should happen |
|---|---|---|
| **shuffle labels within region** | every link between a week's weather and that week's cases, while keeping each region's overall case rate | AUC should fall to ~0.50 |
| **shift weather by 26 weeks** | the alignment between weather and cases | should collapse |
| **replace weather with climatology** | all year-to-year variation; every 2007 week gets the 23-year average for that week | should collapse |
| **use the wrong region's weather** | the place, while keeping the season | should collapse |
| **week of year only** | all weather | this is the seasonality floor |

The first one is the real test. Read its result carefully.

In [2]:
def cv_mean(Xu, yu, factory=ET, per_fold=False):
    aucs, lifts = [], []
    for t in FOLDS:
        tr, te = yr < t, yr == t
        if yu[te].sum() < 5:
            continue
        m = factory(); m.fit(Xu[tr], yu[tr])
        p = m.predict_proba(Xu[te])[:, 1]
        aucs.append(roc_auc_score(yu[te], p))
        lifts.append(average_precision_score(yu[te], p) / yu[te].mean())
    if per_fold:
        return aucs, lifts
    return float(np.mean(aucs)), float(np.mean(lifts))

controls = []
def record(name, auc, lift, expect):
    controls.append(dict(control=name, auc=auc, ap_lift=lift, expected=expect))
    print(f"{name:34s} AUC {auc:.3f}   AP lift {lift:5.2f}x   {expect}")

a, l = cv_mean(X, y);                     record("none (the real model)", a, l, "reference")

ysh = y.copy()
for r in REGIONS:
    m = reg == r
    ysh[m] = rng.permutation(y[m])        # keeps each region's case rate, destroys timing
a, l = cv_mean(X, ysh);                   record("labels shuffled within region", a, l, "should be ~0.50")

shift = np.zeros(len(y), dtype=int)
for r in REGIONS:
    m = reg == r
    idx = np.where(m)[0][np.lexsort((wk[m], yr[m]))]
    shift[idx] = np.roll(idx, 26)
a, l = cv_mean(X[shift], y);              record("weather shifted +26 weeks", a, l, "should collapse")

Xclim = X.copy()
for r in REGIONS:
    for w in range(1, 54):
        m = (reg == r) & (wk == w)
        if m.sum(): Xclim[m] = X[m].mean(axis=0)
a, l = cv_mean(Xclim, y);                 record("weather -> climatology only", a, l, "should collapse")

perm = np.arange(len(y))
swap = dict(zip(REGIONS, REGIONS[1:] + REGIONS[:1]))
for r in REGIONS:
    src = {(yr[i], wk[i]): i for i in np.where(reg == swap[r])[0]}
    for i in np.where(reg == r)[0]:
        perm[i] = src.get((yr[i], wk[i]), i)
a, l = cv_mean(X[perm], y);               record("weather from the wrong region", a, l, "should collapse")

sea = [i for i, f in enumerate(feats) if f in ("week_sin", "week_cos")]
a, l = cv_mean(X[:, sea], y);             record("week of year only", a, l, "seasonality floor")

(ROOT / "outputs").mkdir(exist_ok=True)
with open(ROOT / "outputs" / "expH_controls.csv", "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=list(controls[0])); w.writeheader(); w.writerows(controls)

none (the real model)              AUC 0.821   AP lift  4.55x   reference
labels shuffled within region      AUC 0.803   AP lift  3.51x   should be ~0.50
weather shifted +26 weeks          AUC 0.820   AP lift  3.99x   should collapse
weather -> climatology only        AUC 0.803   AP lift  4.05x   should collapse
weather from the wrong region      AUC 0.802   AP lift  3.63x   should collapse
week of year only                  AUC 0.628   AP lift  1.72x   seasonality floor


## Read the second row again

The case labels were randomly reshuffled inside each region. After that shuffle,
**no week's case count has any relationship to that week's weather.** The model
should be at 0.50.

It scores about 0.81.

It lost roughly 1% of its performance from having its entire outcome variable
randomised. And every other control barely dents it either. Only removing the
weather completely, down to week-of-year, moves the number.

There are no region name columns in this feature table - I checked. But each region
has a distinctive **climate signature**: Kaoping's temperature and humidity profile
is unmistakably Kaoping. So the model learned to recognise the region from its
weather, and then applied that region's historical case rate. Kaoping has a 32%
positive-week rate, Eastern has 0.3%. Ranking a Kaoping week above an Eastern week
is almost free, and in a pooled score it is most of the points.

Let us measure that directly.

## Part 2 - How much of the score is just knowing the region?

Compare the full 185-feature model against the simplest thing imaginable: **one
number per region**, its historical case rate. No weather. No calendar. Six numbers.

In [3]:
P = np.full(len(y), np.nan)
for t in FOLDS:
    tr, te = yr < t, yr == t
    m = ET(); m.fit(X[tr], y[tr])
    P[te] = m.predict_proba(X[te])[:, 1]
ok = ~np.isnan(P)

prev = {r: y[(reg == r) & (yr < 2012)].mean() for r in REGIONS}
P_prev = np.array([prev[r] for r in reg])

pooled = roc_auc_score(y[ok], P[ok])
region_only = roc_auc_score(y[ok], P_prev[ok])

print(f"full 185-feature weather model, pooled AUC   {pooled:.3f}")
print(f"one constant per region, no weather at all   {region_only:.3f}")
print(f"everything the weather data adds             {pooled - region_only:+.3f}")

# within-region AUC: score each region on its own
aucs, wts, per = [], [], {}
for r in REGIONS:
    m = ok & (reg == r)
    if y[m].sum() < 10 or y[m].sum() == m.sum():
        per[r] = None; continue
    a = roc_auc_score(y[m], P[m])
    per[r] = a; aucs.append(a); wts.append(y[m].sum())
within = float(np.average(aucs, weights=wts))

print(f"\nwithin-region AUC (positives-weighted)       {within:.3f}")
print(f"\nskill above chance, pooled {pooled-0.5:.3f}   within region {within-0.5:.3f}")
print(f"share of the pooled skill that is just between-region: "
      f"{100*(1-(within-0.5)/(pooled-0.5)):.0f}%")

print(f"\n{'region':10s} {'positives':>10s} {'within-region AUC':>18s} {'base rate':>10s}")
for r in REGIONS:
    m = ok & (reg == r)
    s = f"{per[r]:.3f}" if per[r] else "too few cases"
    print(f"{r:10s} {int(y[m].sum()):10d} {s:>18s} {y[m].mean():10.3f}")

full 185-feature weather model, pooled AUC   0.821
one constant per region, no weather at all   0.817
everything the weather data adds             +0.004

within-region AUC (positives-weighted)       0.637

skill above chance, pooled 0.321   within region 0.137
share of the pooled skill that is just between-region: 57%

region      positives  within-region AUC  base rate
Central            61              0.639      0.083
Eastern             2      too few cases      0.003
Kaoping           236              0.646      0.323
Northern           11              0.526      0.015
Southern           54              0.621      0.074
Taipei             18              0.629      0.025


## Part 3 - The decisive test

Within one region, in one week of the year, does **this** year's weather tell you
anything the calendar does not already tell you?

To answer that we need a proper baseline, and the right one is the **climatology**:
for each region and each week of the year, how often has a case appeared in that week
historically? That is a lookup table of 6 x 52 numbers. It uses no weather.

We also build **anomaly** features - each feature with its region-and-week average
subtracted - so one model can only see how unusual this year is, with the seasonal
pattern stripped out. If weather matters, that model should beat the lookup table.

Everything is computed from training years only, so nothing peeks at the future.
This cell takes a few minutes.

In [ ]:
SEASON_IDX = [i for i, f in enumerate(feats) if f in ("week_sin", "week_cos")]
WEATHER_IDX = [i for i in range(len(feats)) if i not in SEASON_IDX]

def climatological_risk(train_mask, test_mask, halfwidth=2):
    """P(case) for this region in this week of the year, from training years only."""
    out, ti = [], np.where(test_mask)[0]
    for i in ti:
        wl = [((wk[i] - 1 + d) % 52) + 1 for d in range(-halfwidth, halfwidth + 1)]
        m = train_mask & (reg == reg[i]) & np.isin(wk, wl)
        out.append(y[m].mean() if m.sum() >= 10 else y[train_mask & (reg == reg[i])].mean())
    return np.array(out)

def anomalies(train_mask):
    """Subtract the region x week-of-year mean, learned on training years only."""
    A = X.copy()
    for r in REGIONS:
        for w in range(1, 54):
            tr = train_mask & (reg == r) & (wk == w)
            tgt = (reg == r) & (wk == w)
            if tr.sum() and tgt.sum():
                A[tgt] = X[tgt] - X[tr].mean(axis=0)
    return A

NAMES = ["region base rate", "climatology (region x week)", "season only",
         "raw weather features", "weather anomalies", "anomalies + season"]
PREDS = {k: np.full(len(y), np.nan) for k in NAMES}

for t in FOLDS:
    tr, te = yr < t, yr == t
    if y[te].sum() < 5:
        continue
    base = {r: y[tr & (reg == r)].mean() for r in REGIONS}
    PREDS["region base rate"][te] = [base[r] for r in reg[te]]
    PREDS["climatology (region x week)"][te] = climatological_risk(tr, te)

    m = ET(); m.fit(X[np.ix_(tr, SEASON_IDX)], y[tr])
    PREDS["season only"][te] = m.predict_proba(X[np.ix_(te, SEASON_IDX)])[:, 1]

    m = ET(); m.fit(X[tr], y[tr])
    PREDS["raw weather features"][te] = m.predict_proba(X[te])[:, 1]

    A = anomalies(tr)
    m = ET(); m.fit(A[np.ix_(tr, WEATHER_IDX)], y[tr])
    PREDS["weather anomalies"][te] = m.predict_proba(A[np.ix_(te, WEATHER_IDX)])[:, 1]

    m = ET(); m.fit(A[tr], y[tr])
    PREDS["anomalies + season"][te] = m.predict_proba(A[te])[:, 1]
    print("fold", t, "done")

In [ ]:
ok = ~np.isnan(PREDS["raw weather features"])
idx_by_reg = {r: np.where(ok & (reg == r))[0] for r in REGIONS}

def within_region(p):
    aucs, lifts, wts = [], [], []
    for r in REGIONS:
        m = ok & (reg == r)
        if y[m].sum() < 10 or y[m].sum() == m.sum():
            continue
        if len(set(p[m].round(12).tolist())) < 2:
            aucs.append(0.5); lifts.append(1.0); wts.append(y[m].sum()); continue
        aucs.append(roc_auc_score(y[m], p[m]))
        lifts.append(average_precision_score(y[m], p[m]) / y[m].mean())
        wts.append(y[m].sum())
    return float(np.average(aucs, weights=wts)), float(np.average(lifts, weights=wts))

def boot_ci(p, n=1500):
    vals = []
    for _ in range(n):
        aucs, wts = [], []
        for idx in idx_by_reg.values():
            if len(idx) == 0: continue
            k = idx[rng.integers(0, len(idx), len(idx))]
            if y[k].sum() < 8 or y[k].sum() == len(k): continue
            if len(set(p[k].round(12).tolist())) < 2: continue
            aucs.append(roc_auc_score(y[k], p[k])); wts.append(y[k].sum())
        if aucs: vals.append(np.average(aucs, weights=wts))
    return np.percentile(vals, [2.5, 97.5])

print(f"{'predictor':30s} {'pooled':>8s} {'WITHIN REGION':>14s} {'95% CI':>18s} {'AP lift':>8s}")
final = []
for name in NAMES:
    p = PREDS[name]
    pooled_a = roc_auc_score(y[ok], p[ok]) if len(set(p[ok].round(12).tolist())) > 1 else 0.5
    w, lift = within_region(p)
    lo, hi = boot_ci(p)
    print(f"{name:30s} {pooled_a:8.3f} {w:14.3f} {f'[{lo:.3f}, {hi:.3f}]':>18s} {lift:8.2f}")
    final.append(dict(predictor=name, pooled_auc=pooled_a, within_auc=w,
                      ci_lo=float(lo), ci_hi=float(hi), within_ap_lift=lift))

with open(ROOT / "outputs" / "expK_within_region.csv", "w", newline="") as f:
    w_ = csv.DictWriter(f, fieldnames=list(final[0])); w_.writeheader(); w_.writerows(final)

# paired test: does the weather model beat the lookup table?
a, b = PREDS["anomalies + season"], PREDS["climatology (region x week)"]
diffs = []
for _ in range(3000):
    pa, pb, wts = [], [], []
    for idx in idx_by_reg.values():
        if len(idx) == 0: continue
        k = idx[rng.integers(0, len(idx), len(idx))]
        if y[k].sum() < 8 or y[k].sum() == len(k): continue
        if len(set(b[k].round(12).tolist())) < 2: continue
        pa.append(roc_auc_score(y[k], a[k])); pb.append(roc_auc_score(y[k], b[k]))
        wts.append(y[k].sum())
    if pa:
        diffs.append(np.average(pa, weights=wts) - np.average(pb, weights=wts))
diffs = np.array(diffs)
lo, hi = np.percentile(diffs, [2.5, 97.5])
pval = 2 * min((diffs <= 0).mean(), (diffs >= 0).mean())
print(f"\nweather anomalies + season, minus the climatology lookup table:")
print(f"  {diffs.mean():+.3f}   95% CI [{lo:+.3f}, {hi:+.3f}]   p = {pval:.4f}")

## The finding

> **Within a region, the year-to-year weather anomaly carries no usable information
> about whether a given week will produce a melioidosis case. The predictive skill of
> climate-based melioidosis models is the seasonal climatology plus the region's base
> rate, and nothing else.**

The best predictor of all is a lookup table of 312 numbers using no weather data.
Every machine learning model is worse, and the model restricted to genuine weather
anomalies is **significantly** worse, p around 0.001. Not "no better". Worse.

## Why this explains everything else in the project

| earlier result | explanation |
|---|---|
| 141 mechanistic features added nothing | there was no weather signal for them to improve |
| transfer to another region fails | what transfers is a region-recognition shortcut |
| the shortcut is *inverted* in Central Taiwan | a shortcut can point the wrong way somewhere new |
| climatic distance did not predict transfer failure (r = -0.098) | the failure was never about climate |
| local target data fixed calibration but not ranking | the ranking was never weather-driven |
| SHAP's top feature is day-night temperature range | that is a geography variable, not a mechanism |

Nine experiments that looked like nine dead ends are one finding.

## What to put in the paper

1. Report **within-region** AUC, not pooled AUC, whenever data spans multiple places.
2. Always include a **climatology baseline**. A model that cannot beat a week-of-year
   lookup table has not demonstrated a climate effect.
3. Always run the **shuffled-label control**. It is five lines of code and it would
   have caught this immediately.
4. Published AUCs of 0.89-0.95 for melioidosis climate models are consistent with
   measuring region identity and seasonality. That is not an accusation of error -
   it is a statement that the standard evaluation cannot distinguish the two, and
   these three checks can.

Notebook 06 shows what happens on transfer. Notebook 07 shows what *does* survive.